# report04 — 조명원 — **WiFi / LTE / 5G 파형**

**핵심.** 패시브 레이더는 자기 송신기가 없어 남의 **상시 신호**를 빌려 드론을 비춘다 — 어떤 상시 신호가 좋은 조명원인가를 WiFi·LTE·5G 세 후보로 한 챔버에서 가른다. 결론: 넓고($\Delta R_b$ 16.7 m) 자주(1000 Hz) 나오는 **LTE CRS 가 좁고($\Delta R_b$ 41.6 m) 드문(20 ms) 5G SSB 보다 낫다** — 세대가 최신이라고 조명 품질이 좋은 건 아니다.

| 이 리포트의 척추 |  |
|---|---|
| **① Sionna 의 공백** | **Sionna PHY 는 5G NR OFDM 을 3GPP 규격대로 생성**한다(`nr.CarrierConfig` 가 μ·SCS·슬롯·CP 를 준다). 그러나 **`nr.CarrierConfig` 는 5G NR 전용이라 LTE 파형·뉴머롤로지는 TS 36.211 로 자작**하고, **WiFi(802.11) 파형도 기본 제공하지 않으며**, 패시브가 실제로 상관에 쓰는 **상시 기준신호(LTE CRS·5G SSB)의 자원격자 배치**도 없다 — '채널 전대역'이 아니라 그 기준신호가 격자에서 실제로 켜는 **점유대역 $B_{ref}$** 가 눈을 정한다(§2). |
| **② 선행 연구의 방식** | 기회신호(illuminator-of-opportunity) 패시브 레이더 문헌이 고르는 조명원이 정확히 이 상시 기준신호들이다 — **LTE450** 하향링크 드론 검출(Demissie 외, *2024 International Radar Conference (RADAR)*, DOI 10.1109/RADAR58436.2024.10993905 — 원문 확인), 셀룰러 하향링크 멀티스태틱 도플러 추적(arXiv:2509.25732 — 프리프린트, 원문 미확인), **5G SSB** 저고도 드론 검출·측위(Jopanya & Osorio, *IEEE SPAWC* 2025, DOI 10.1109/SPAWC66079.2025.11143316) (§3). |
| **③ 쓴 라이브러리·결합** | **5G NR 뉴머롤로지는 Sionna PHY `nr.CarrierConfig`(TS 38.211 구현) 에 물어**(중복 구현 회피), **LTE·WiFi 파형과 상시 기준신호(CRS/SSB) 격자는 3GPP TS 36.211/38.211·IEEE 802.11ac 표에서 자작**(`src/waveforms.py`)한다. 점유대역·PRF 를 격자에서 읽어 $\Delta R_b$=c/$B_{ref}$·$V_{b,max}$=PRF·λ/2 로 환산한다 — 거리축이 두 경로의 합 $R_b$ 이므로 속도축도 같은 거리합 좌표에 둔다(모노스태틱 등가는 그 절반)(§4). |
| **④ 검증** | 파형이 정말 규격대로인지는 **Sionna PHY 로 교차대조(report05)** — 대역폭·뉴머롤로지·상시성을 규격 대비 확인. 본문 숫자는 유휴 셀 격자에서 실측한 값(WiFi 76.6·LTE 18.0·SSB 7.2 MHz). |

---


## 📋 이 결과가 어디서 어떻게 나왔나

> 이 절은 **직접 참여하지 않은 사람도 출처를 따라가고 재현할 수 있도록** 넣었습니다. 버전·GPU 는 노트북 생성 시점에 **실제로 읽어온 값**입니다.

### 1️⃣ 무엇을 참고했나

| 항목 | 출처 | 성격 |
|---|---|---|
| LTE CRS / 5G SSB / WiFi VHT-LTF 자원격자 배치 | 3GPP TS 36.211 · 3GPP TS 38.211 · IEEE 802.11ac → `src/waveforms.py`. 조사 근거는 `docs/waveform_research.json` | 🔴 우리 구현 (Sionna 에 WiFi/LTE/SSB **자원격자가 없다**) |
| 5G NR 뉴머롤로지 (μ·SCS·슬롯·CP) | **`sionna.phy.nr.CarrierConfig`** — Sionna 에게 물어본 값 (표를 손으로 안 짬) | 🟢 라이브러리 (3GPP TS 38.211 구현) |
| $\Delta R_b$ · $V_{b,max}$ 폐형식 | $\Delta R_b$ = c/$B_{ref}$ · $V_{b,max}$ = PRF·λ/2 — **둘 다 거리합(바이스태틱) 좌표**($f_d=\dot R_b/\lambda$). 모노스태틱 등가로 환산하면 각각 c/2B · PRF·λ/4 — 교과서 레이더 공식 | 📐 해석식 |

### 2️⃣ 어떤 도구가 무엇을 했나 — **Sionna 내부인가, 우리가 짠 건가**

| 도구 | 하는 일 | 어디서 도는가 |
|---|---|---|
| `sionna-phy` | Sionna PHY (`ofdm`/`nr`/`channel`) — OFDM 변복조 · 3GPP 뉴머롤로지 · RT 경로를 신호에 적용 | 🟢 **Sionna 내부** (PyTorch 백엔드, GPU) |
| `matplotlib` | matplotlib — 도표·그래프 | 🔴 **별도** (CPU). 계산 결과를 *그리기만* 한다 |

> 🔑 **이 구분이 이 프로젝트에서 가장 자주 오해받는 지점입니다.**
> - **전파**(경로·지연·도플러·렌더·라디오맵)는 🟢 **Sionna 가** 합니다.
> - **표적 RCS** 는 🟡 우리가 얹은 **PO(물리광학 표면적분)** 가 냅니다 — Sionna 기본 solver 엔 이 산란적분이 없어 경로 이득만 줄 뿐 RCS 를 못 내기 때문입니다. 광선을 쏴 조명면·가림을 찾는 **SBR** 은 Sionna 의 **Mitsuba 3 엔진을 그대로** 쓰고, 그 위에 **PO 적분만 우리가** 얹습니다(SBR+PO).
> - **레이더 신호처리**(ECA/CFAR)는 🔴 우리가 짰습니다 — Sionna 에 레이더 DSP 가 없습니다.

### 3️⃣ 라이브러리 (실행 시점 **실측** 버전)

| 라이브러리 | 버전 | 무엇에 쓰나 |
|---|---|---|
| `sionna` | 2.0.1 | 광선추적(RT) + PHY(OFDM/NR/채널) — **이 프로젝트의 중심** |
| `numpy` | 2.5.0 | 수치 계산 전반 |
| `scipy` | 1.18.0 | 스플라인(단면 보간·암 경로) · STFT(스펙트로그램) |
| `matplotlib` | 3.11.0 | 도표 |

### 4️⃣ 어디서 돌렸나

- **Python** 3.12.13 · Linux 5.15.0-136-generic
- **GPU** — `src/gpu.py` 가 **여유 메모리를 보고 자동 선택**합니다 (하드코딩 없음):
  - 0, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
  - 1, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
  - 2, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
  - 3, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
- `CUDA_VISIBLE_DEVICES` = (고정 안 함 — src/gpu.py 가 여유 메모리 보고 자동 선택)

- **계산 비용**: 이 리포트는 무거운 재측정을 하지 않는다 — 파형 격자를 만들어 제원을 읽는 것은 CPU 초 단위. 그림은 report2 파이프라인이 이미 남긴 것을 재사용한다. 노트북 생성도 초 단위.

### 5️⃣ 어떻게 다시 돌리나 (재현)

```bash
cd /home/yunjung/workspace/sionna2

# 파형 제원을 만들어서 재고 그림·JSON 을 남긴다 (report2 파이프라인 재사용, 재측정 없음)
~/.venvs/py312/bin/python src/viz_report2.py        # 측정 + 그림 + JSON
~/.venvs/py312/bin/python src/make_notebook04.py    # JSON -> report04.ipynb

# 파형 제원을 직접 눈으로 확인하고 싶으면:
~/.venvs/py312/bin/python src/waveforms.py          # 표준별 B_ref · PRF · ΔR · v_max
```

### 6️⃣ 본문 숫자는 어디서 오나

이 노트북의 **숫자는 손으로 적지 않았습니다.** 측정 스크립트가 JSON 을 남기고, 노트북 생성기(`src/make_notebook*.py`)가 그 JSON 을 읽어 본문에 주입합니다. → **그림과 글이 어긋날 수 없습니다.** 숫자가 이상하면 JSON 을 보세요.

### 7️⃣ 무엇이 산출되나

| 산출물 | 무엇 |
|---|---|
| `outputs/report2_waveform_rcs.json` | **이 리포트의 모든 파형 숫자.** 측정 원본 |
| `outputs/figures/report2_ref_signal.png` | §2 넓이→거리 · 반복→속도 예산 그림 |
| `outputs/figures/report2_resource_grid.png` | §2 자원격자 사진 (WiFi/LTE/5G × 유휴/풀로드) |
| `outputs/figures/report2_occupancy.png` | §2 점유율은 늘어도 거리분해능은 안 늘더라 |

### 8️⃣ ⚠️ 믿으면 안 되는 것 (신뢰 경계)

> 정직함이 이 프로젝트의 규칙입니다. **아래는 이 리포트가 보장하지 않는 것들입니다.**

- **§2 의 ΔR·$v_{max}$ 는 '유휴 셀'을 전제한다.** 유휴 셀은 상시 기준신호만 내보내므로 패시브가 기댈 수 있는 **가장 정직한 기본선**이다. 부하가 걸린 셀에서 수신기가 송신 파형 전체를 직접 받아 기준으로 쓸 수 있다면(captured full-waveform) 대역은 채널 전대역까지 넓어질 수 있으나, 그건 **다른 전제**다 — 두 경우를 섞어 인용하지 말 것.
- **PRS 로 넓힌 5G 수치는 낙관적 상한이다.** PRS 는 상시 신호가 아니라 **측위 세션이 설정돼야** 켜지는 옵션이며, 남의 셀을 빌려 쓰는 패시브 수신기는 그것이 켜져 있다고 가정할 수 없다. 그래서 §2 의 기본선은 SSB(7.2 MHz)이지 PRS(전대역)가 아니다.
- **WiFi 의 PRF 1 kHz 는 '혼잡한 AP' 대표값이다.** WiFi 는 LTE CRS·5G SSB 처럼 늘 나오는 상시 신호가 아니라 **트래픽이 있을 때만** 촘촘하다 — 트래픽이 없으면 비콘(약 100 ms 주기)만 남아 반복이 훨씬 느려진다. 그래서 WiFi 패시브 센싱 연구들은 흔히 **의도적으로 트래픽을 유발**(예: 대상 AP 에 ping flood)해 조명 신호를 확보하는데, 이는 순수 '남의 상시 신호'라기보다 관측자가 신호를 만들어내는 쪽에 가깝다. 드론 탐지에서 WiFi 가 주 조명원으로 쓰이는 사례도 드물다 — 이 리포트가 WiFi 를 후보로 넣되 상시성 한계를 함께 못박는 이유다.
- **이 리포트는 파형이 무엇을 주는가(대역·반복·분해능)만 다룬다.** 그 파형이 정말 규격대로 만들어졌는지의 증명은 → report05, 표적이 얼마나 밝은지(RCS)는 → report06 소관이다.

### 9️⃣ 앞뒤 리포트

| 리포트 | 관계 |
|---|---|
| [report03](report03.ipynb) — 표적 검증 | **앞 리포트.** '무엇을 볼 것인가'(드론)를 확정했다. 여기서는 '무엇으로 볼 것인가'(조명원) |
| **report04 (여기)** — 조명원 · 파형 | 패시브가 빌려 쓰는 상시 신호 3종과 그 장단점 — 넓이(거리)·반복(속도) |
| [report05](report05.ipynb) — 파형 검증 | **다음 리포트.** 여기서 만든 파형이 정말 3GPP/IEEE 규격대로인지 Sionna 로 대조한다 |

<details><summary><b>🔤 용어집 — 모르는 말이 나오면 여기</b> (클릭)</summary>

| 용어 | 뜻 |
|---|---|
| **패시브 레이더** | 자기 송신기 없이 **남의 신호(방송·통신)의 반사**로 표적을 보는 레이더 |
| **조명원(illuminator)** | 패시브 레이더가 빌려 쓰는 '남의 송신기' — 기지국·와이파이 AP 등 |
| **기준신호(reference signal)** | 패시브가 상관을 걸 때 쓰는 '미리 아는 신호'. 표준마다 상시 하나 |
| **상시(always-on) 신호** | 임의의 셀이 **언제나** 내보낸다고 믿을 수 있는 신호. LTE=CRS, 5G=SSB (WiFi 프리앰블은 트래픽이 있을 때만 반복돼 엄밀히는 상시 아님 — §8) |
| **CRS** | LTE Cell-specific Reference Signal — 매 서브프레임(1 ms)·채널 전대역에 상시 나오는 셀 기준신호 |
| **SSB** | 5G NR SS/PBCH Block — 유휴 gNB 가 늘 내보내는 유일한 신호. 좁고(중앙 20 RB) 드물다(20 ms 주기) |
| **VHT-LTF** | WiFi 802.11ac 패킷 앞머리(프리앰블)의 롱 트레이닝 필드. 어떤 패킷에도 붙지만 반복은 트래픽에 의존 |
| **점유대역 $B_{ref}$** | 기준신호가 자원격자에서 **실제로 켜는** 주파수 폭. 채널 전대역과 다르다 |
| **거리분해능 ΔR** | 두 표적을 거리로 가르는 최소 간격. **바이스태틱** $\Delta R_b$=c/$B_{ref}$ (우리 시스템), 모노스태틱 등가는 c/2B(절반). 넓은 신호일수록 촘촘 |
| **PRF** | pulse repetition frequency — 기준신호가 **초당 몇 번** 반복되나. 자주일수록 빠른 표적을 봄 |
| **무모호 속도 $V_{b,max}$** | 헷갈리지 않고 볼 수 있는 최대 **거리합 변화율** $\dot R_b$. $V_{b,max}$ = PRF·λ/2 — 거리축 $\Delta R_b$=c/$B_{ref}$ 와 **같은 좌표계**. 모노스태틱 등가는 그 절반(PRF·λ/4) |
| **자원격자(resource grid)** | 가로=시간·세로=주파수의 모눈종이. OFDM 신호는 이 칸을 채워 만든다 |
| **RE (resource element)** | 자원격자의 **한 칸** (한 부반송파 × 한 OFDM 심볼) |
| **OFDM** | 부반송파 수백 개에 데이터를 잘게 나눠 싣는 변조 방식. 통신 신호의 기본 골격 |
| **PRS** | Positioning Reference Signal — **측위 세션이 설정돼야** 켜지는 옵션. 상시 신호가 아니다 |
| **도플러 접힘(aliasing)** | 반복이 너무 드물어 빠른 표적의 속도를 헷갈리는 현상. $v_{max}$ 를 넘으면 생긴다 |

</details>

---


## §1. Sionna 가 주는 파형과 안 주는 파형 — 패시브가 빌리는 상시 신호

패시브 레이더는 자기 송신기 없이 **남의 신호의 반사**로 표적을 본다. 송신기를 빌려 쓰므로 두 제약이 따라붙는다.

1. **미리 아는 신호여야 한다.** 튕겨 온 신호가 표적인지 알아보려면 '원래 신호'와 상관해야 하는데, 통신 신호가 나르는 **데이터는 매 순간 바뀌어** 수신기가 미리 알 수 없다. 내용이 고정된 **기준신호(reference signal)** 만 상관에 쓸 수 있다.
2. **아무 셀이나 늘 내보내야 한다.** 특정 조건에서만 켜지는 신호는 하필 그 순간 그 셀이 안 켜면 표적을 놓친다. 임의의 셀이 **언제나** 내보낸다고 믿을 수 있는 **상시(always-on) 신호** 여야 한다.

이 두 조건을 동시에 만족하는 신호는 표준마다 **딱 하나씩**이다:

| 표준 | 상시 기준신호 | 왜 이것뿐인가 |
|---|---|---|
| **LTE** | **CRS** (Cell-specific Reference Signal) | 매 서브프레임(1 ms)·채널 전대역에 늘 뿌린다 |
| **5G NR** | **SSB** (SS/PBCH Block) | NR 에는 **CRS 같은 상시 전대역 신호가 없다.** 유휴 기지국이 늘 내보내는 건 SSB 뿐 |
| **WiFi** | **프리앰블 (VHT-LTF)** | 모든 패킷 앞머리에 붙는다 — 단 **반복 횟수가 트래픽에 의존** |

여기서 **Sionna PHY 의 공백**이 드러난다. Sionna PHY(`nr`/`ofdm`/`channel`)는 **5G NR** 의 OFDM 파형을 3GPP 규격대로 만들고 뉴머롤로지(μ·SCS·슬롯·CP)도 `nr.CarrierConfig` 가 TS 38.211 을 구현해 주지만 — **① `nr.CarrierConfig` 는 5G NR 전용이라 LTE 뉴머롤로지(15 kHz SCS)는 TS 36.211 로 자작**하고, **② WiFi(802.11) 파형은 기본 제공하지 않으며**, **③ 패시브가 실제로 상관에 쓰는 상시 기준신호(CRS·SSB)의 자원격자 배치**(어느 칸을 켜는가)도 API 로 바로 주지 않는다. 그런데 조명원의 성능을 정하는 것은 '채널 전대역'이 아니라 그 기준신호가 격자에서 **실제로 켜는 점유대역 $B_{ref}$** 다(§2). 그래서 우리는 세 표준의 상시 기준신호 격자를 규격표 위에 직접 세워 (§4) 그 $B_{ref}$·반복주기를 읽는다.

> ⚠️ **PRS(측위 기준신호)는 상시 신호가 아니다.** 넓은 대역을 쓰지만 **측위 세션이 설정돼야** 켜지는 옵션이라, 남의 셀을 빌려 쓰는 패시브 수신기는 그것이 켜져 있다고 가정할 수 없다. 이 리포트의 기본선은 어디까지나 위 3개의 **상시** 신호다.

---
## §2. 무엇을 보는가 — 넓이(거리)와 반복(속도), 자원격자로 확인

빌린 신호의 **주파수 넓이**와 **반복 빈도**가 각각 거리·속도를 얼마나 잘 보는지를 정한다. 유휴 셀 격자에서 읽은 값으로 세 조명등을 나란히 세운다.

### 2.1 넓을수록 거리를 잘 가른다

> 손전등 빛줄기가 굵으면 두 물체가 한 덩어리로 뭉쳐 보이고, 가늘고 날카로울수록 둘을 따로 짚는다. 레이더에서 이 '날카로움'을 정하는 게 신호의 **주파수 넓이**다.

두 표적을 거리로 가르는 최소 간격이 **거리분해능 ΔR** 이다(우리는 **바이스태틱**이라 $\Delta R_b$=c/$B_{ref}$; 모노스태틱 등가는 절반). 결정적인 것은 채널이 차지한 전체 대역이 **아니라**, 기준신호가 자원격자에서 **실제로 켜는 주파수 폭** — **점유대역 $B_{ref}$** 다.

$$\Delta R_b = \frac{c}{B_{ref}}\quad(\text{바이스태틱; 모노 등가 } c/2B)$$

$B_{ref}$ 가 넓을수록 ΔR 이 작아져(=촘촘해져) 가까이 붙은 두 물체를 갈라 본다.

### 2.2 자주 나올수록 빠른 표적을 본다

표적이 움직이면 되돌아오는 신호의 주파수가 살짝 밀린다(도플러). 이 밀림으로 속도를 재는데, 기준신호가 **초당 몇 번 반복되나**(PRF)가 헷갈리지 않고 볼 수 있는 최고 속도 — **무모호 속도** 를 정한다. 반복이 드물면 그보다 빠른 표적은 **도플러가 접혀(aliasing)** 엉뚱한 속도로 읽힌다.

속도축은 **거리축과 같은 좌표계**여야 한다. 우리 RD 맵의 거리축은 두 경로의 합 $R_b$ 이고, 그 좌표에서 도플러는 $f_d=\dot R_b/\lambda$ 다(검출 체인이 쓰는 규약도 같다 — `benchmark/geometry.py:135` 의 `fd = v @ (u1+u2) / lam`, 계수 2 없음). 도플러 표본화 한계 $|f_d|\le \mathrm{PRF}/2$ 를 그대로 옮기면:

$$V_{b,max} = \frac{\mathrm{PRF}\cdot\lambda}{2}\quad(\text{거리합 } \dot R_b \text{ 기준; 모노 등가 } \mathrm{PRF}\cdot\lambda/4)$$

$\dot R_b = -\,\mathbf{v}\cdot(\hat u_1+\hat u_2)$ 이므로 **표적 속력**으로 환산한 접힘 문턱은 기하에 따라 달라진다 — 두 경로가 나란한 극한(모노스태틱 등가)에서 가장 엄격해 PRF·λ/4 가 된다.

### 2.3 유휴 셀에서 세 조명등을 나란히

5G NR 뉴머롤로지(μ·SCS·슬롯·CP)는 손으로 짜지 않고 **Sionna PHY 의 `CarrierConfig`** 가 3GPP TS 38.211 을 구현한 값을 그대로 받는다. 각 표준의 상시 기준신호를 그 규격 격자 위에 올려 두고, 격자가 **실제로 켜는** $B_{ref}$·PRF 를 읽어 $\Delta R_b$·$V_{b,max}$ 를 교과서 폐형식으로 구한다 (아래 숫자는 손으로 적은 게 아니라 **격자에서 읽은** 값이다):

| 표준 | 기준신호 | 채널 대역 | **$B_{ref}$** | **$\Delta R_b$** | **PRF** | **$V_{b,max}$** | 모노 등가 $v_{max}$ |
|---|---|---|---|---|---|---|---|
| WiFi 802.11ac | VHT-LTF | 80 MHz | **76.6 MHz** | **3.9 m** | 1000 Hz | 28.8 m/s | 14.4 m/s |
| LTE Rel-9 | CRS | 18 MHz | **18.0 MHz** | **16.7 m** | 1000 Hz | 81 m/s | 41 m/s |
| **5G NR Rel-16** | **SSB** | 98 MHz | **7.2 MHz** | **41.6 m** | **50 Hz** | **2.14 m/s** | 1.07 m/s |

![reference signal budget](outputs/figures/report2_ref_signal.png)

*(그림 (a): 회색 = 채널 전체 대역, 색 = 패시브가 실제로 쓰는 기준신호 대역. 5G 만 둘이 크게 다르다. (b): 그 대역이 정하는 거리분해능. (c): 거리(가로)·속도(세로)를 한 점으로 — 오른쪽·아래일수록 나쁨. 그림 (c) 의 속도축은 라벨대로 **모노스태틱 등가 $v_{max}$**(표 마지막 열)이다 — 거리합 좌표로 읽으려면 두 배 하면 된다.)*

**세 줄로 읽는 법:**
- **WiFi** 는 기준신호가 이미 넓어(**76.6 MHz**) 거리를 가장 촘촘히 본다(**$\Delta R_b$ 3.9 m**). 다만 5 GHz 대라 도달 거리가 짧고, 반복이 트래픽에 의존한다.
- **LTE** 는 CRS 가 채널 전대역(**18.0 MHz**)에 매 1 ms 뿌려져 거리(**$\Delta R_b$ 16.7 m**)·속도(**$V_{b,max}$ 81 m/s**) 모두 균형이 좋다.
- **5G SSB** 는 채널이 98 MHz 나 되는데도 상시 켜는 건 중앙 20 RB **7.2 MHz** 뿐 — 거리가 거칠고(**$\Delta R_b$ 41.6 m**), 20 ms 마다 한 번만 나와 속도도 약하다(**$V_{b,max}$ 2.14 m/s**).

In [ ]:
# §2 재현 — waveforms.py 가 격자에서 읽은 속성을 그대로 출력 (본문 숫자에 하드코딩 없음)
import sys; sys.path.insert(0, 'src')
from waveforms import always_on_waveforms

print(f"{'표준':16s} {'기준신호':9s} {'B_ref':>10} {'ΔR_b':>8} {'PRF':>9} {'V_b,max':>10} {'v_max(모노)':>11}")
for k, wf in always_on_waveforms().items():          # 유휴 셀 = 상시 기준신호만
    # waveforms.py 의 v_unambiguous_ms 는 **모노스태틱 등가**(PRF·λ/4).
    # 거리축이 거리합 ΔR_b=c/B_ref 이므로 속도축도 거리합 좌표(PRF·λ/2)로 맞춘다.
    vb = 2.0 * wf.v_unambiguous_ms
    print(f'{wf.name:16s} {wf.ref_name:9s} {wf.ref_bw_hz/1e6:7.2f}MHz '
          f'{wf.range_resolution_m:6.2f}m {wf.pilot_rate_hz:7.0f}Hz {vb:8.2f}m/s {wf.v_unambiguous_ms:9.2f}m/s')

### 2.4 자원격자로 본 세 신호

통신 신호는 **자원격자(가로=시간, 세로=주파수인 모눈종이)** 의 칸(RE)을 채워 만든다. 아래 그림에서 **위 줄은 유휴 셀**(상시 기준신호만), **아래 줄은 풀로드 셀**(데이터까지 꽉 채운 상태)이다.

![resource grid](outputs/figures/report2_resource_grid.png)

위 줄만 보면 된다 — 패시브가 쓸 수 있는 건 그것뿐이다. **WiFi 프리앰블**은 세로(주파수)로 꽉 차 있고(넓다), **LTE CRS** 도 전대역에 흩뿌려져 있는데, **5G SSB** 만 세로로 **가운데 작은 블록**에 몰려 있다 — 이 좁음이 곧 거친 거리분해능이다. 아래 줄(풀로드)에서 회색으로 꽉 찬 칸(PDSCH/DATA)은 **데이터**다. 에너지는 많지만 수신기가 **모르는 내용**이라 상관에 못 쓴다 — 그래서 셀이 바빠져도 패시브가 쓸 '아는 신호'의 넓이는 거의 안 늘어난다.

### 2.5 셀이 바빠져도 거리분해능은 안 늘더라

'셀이 데이터로 꽉 차면 신호가 넓어져 거리를 더 잘 보지 않을까?' — 만들어서 재보면 그렇지 않다.

![occupancy](outputs/figures/report2_occupancy.png)

격자 점유율(왼쪽)은 유휴→풀로드로 5G 기준 1.4% → 80% 로 수십 배 뛰지만, **거리분해능(오른쪽)은 WiFi·LTE 는 거의 안 움직인다** — 상시 기준신호가 **이미 넓기** 때문이다. 늘어난 건 상관에 못 쓰는 데이터 에너지뿐이다.

> ⚠️ 5G 만 오른쪽에서 41.6 m → 3.1 m 로 급전환하는데, 그건 **PRS(측위 기준신호)가 켜졌기 때문**이다. PRS 는 상시 신호가 아니라 **측위 세션을 가정한 것**이므로, 이 급전환은 **낙관적 상한**이지 패시브가 늘 기댈 수 있는 기본선이 아니다. 기본선은 어디까지나 SSB(7.2 MHz)다.

### 2.6 5G 가 불리한 이유 — 좁고, 드물다

**① 좁다 (거리가 거칠다).** SSB 는 채널 한가운데 20 RB = **7.2 MHz** 만 켠다. 그래서 $\Delta R_b$ = **41.6 m** — 42 m 안쪽에 있는 두 물체는 **한 덩어리**로만 보인다. 채널 자체는 **98 MHz** 나 되는데 패시브가 상시로 쓸 수 있는 건 그중 **7.3%** 뿐이다.

**② 드물다 (빠른 표적을 놓친다).** SSB 는 SS 버스트가 **20 ms 주기**로만 나온다 → PRF **50 Hz** → $V_{b,max}$ **2.14 m/s**(거리합 기준). 거리합 변화율 $\dot R_b$ 는 송신·수신 두 경로 변화율의 **합**이라, 사람 걸음(약 1.4 m/s) 정도의 표적도 기하에 따라 이 한계를 넘어 도플러가 접힌다 — 두 경로가 나란한 극한에서는 표적 속력 **1.07 m/s** 만 넘어도 접힌다.

### 2.7 대비 — 구세대 LTE 가 오히려 좋은 조명등

LTE CRS 는 **전대역**(18.0 MHz)을 켜고 **매 서브프레임(1 kHz)** 나온다. 그래서 거리·속도 두 축 모두 5G 를 앞선다:

| 축 | 무엇이 정하나 | LTE CRS | 5G SSB | LTE 가 유리한 정도 |
|---|---|---|---|---|
| **거리** $\Delta R_b$ | 점유대역 $B_{ref}$ | **16.7 m** (18.0 MHz) | 41.6 m (7.2 MHz) | 약 2.5× 촘촘 |
| **속도** $V_{b,max}$ | PRF | **81 m/s** (1000 Hz) | 2.14 m/s (50 Hz) | 약 38× 빠름 |

통신 세대가 최신이라고 패시브 레이더에 좋은 조명등이 되는 건 아니다. 오히려 5G 는 늘 켜 두는 기준신호가 **좁고 드물어서** 옛 LTE 보다 못하다. 쓸 조명원을 고를 땐 통신 성능이 아니라 **상시 신호의 넓이와 반복**을 봐야 한다.

![.](outputs/renders/anim/spectrum_nr.gif)

<sub>5G NR 스펙트럼 — 상시 기준신호(SSB) 점유 대역이 좁아 거리분해능이 거칠다.</sub>

![.](outputs/renders/anim/spectrum_lte.gif)

<sub>LTE 스펙트럼 — CRS 가 전대역(18.0 MHz)을 켜 거리분해능이 촘촘하다(ΔR_b 16.7 m, 5G SSB 의 약 2.5배 촘촘).</sub>

---
## §3. 선행 연구 — 기회신호 패시브 레이더의 조명원

이 조명원 선택은 우리가 발명한 것이 아니라 **기회신호(illuminator-of-opportunity) 패시브 레이더 문헌이 이미 걸어간 길**이다. 문헌이 드론을 잡을 때 고르는 조명원이 정확히 §1 의 상시 기준신호들이다.

| 조명원 | 선행 연구 | 확립 정도 |
|---|---|---|
| **LTE450** 하향링크 | Demissie 외, *Protection of Critical Infrastructure using LTE450-based Passive Radar* (*2024 International Radar Conference (RADAR)*, DOI 10.1109/RADAR58436.2024.10993905) | peer-reviewed 실측(표적 DJI M210) |
| 셀룰러 하향링크(멀티스태틱) | *Doppler-Based Multistatic Drone Tracking via Cellular Downlink Signals* (arXiv:2509.25732) | 멀티스태틱 도플러 — 프리프린트·원문 미확인 |
| **5G SSB** | Jopanya & Osorio, *Utilizing 5G NR SSB Blocks for Passive Detection and Localization* (*IEEE SPAWC* 2025, DOI 10.1109/SPAWC66079.2025.11143316) | 이론(CRB) 단계 |

**LTE 조명원**으로 드론을 잡는 패시브 레이더는 이미 peer-reviewed 실측으로 확립됐지만(Demissie 외), **5G SSB 만으로** 드론을 보는 쪽은 아직 성능한계(CRB)를 따지는 이론 단계에 가깝다(Jopanya & Osorio) — §2 가 숫자로 보인 5G 의 불리함과 결이 같다.

<sub>서지 주의 — 같은 저자의 후속판 *Drone Detection With a LTE450-Based Passive Radar* (*IET RSN* 2025, DOI 10.1049/rsn2.70092)는 **페이월이라 읽지 못했다**. 그래서 위 표에는 우리가 원문을 연 2024 RADAR 판만 싣고, 처리사슬·수치를 IET 2025 판에 귀속하지 않는다(report10 §3 과 같은 규약). Jopanya 의 식별자는 PDF 표지에 찍힌 DOI 를 1차로 쓴다(arXiv:2504.02641 은 2차 출처 ID).</sub>

이 리포트가 재는 상시 신호의 좁은 대역은 패시브가 늘 기댈 수 있는 **가장 정직한 기본선**이고, 파형 전체를 잡아 쓰는 것은 (수신기가 그럴 수 있을 때의) **다른 전제**다 — 두 경우를 섞어 읽지 않는다.

> **각주 — '상시 파일럿만' vs '파형 전체'는 문헌이 한쪽으로 갈리지 않는다.**
> 
> · Wypich & Zielinski(*Sensors* 2026, AGH + Ericsson)는 5G NR 패시브 레이더에서 사용자 데이터(PDSCH)까지 재구성해 파형 전체를 기준으로 쓰면 검출확률(POD)이 24–32% → 77–78% 로 오른다고 보고한다(ECA+ → CA-CFAR, USRP X310). 다만 **표적이 드론이 아니라 차량**(5.8 GHz·33.3 MHz OTA)이고, **우리는 원문 PDF 를 확보하지 못했다**(웹 메타데이터 기준) — 그래서 위 선례표에 넣지 않고 방향성 근거로만 읽는다.
> 
> · 반대 부호의 실측도 있다. Taylor & Poullin(ONERA, *IEEE RADAR* 2023, DOI 10.1109/RADAR54928.2023.10371153)은 **UAV 표적**의 LTE 패시브 레이더에서 원문 축자로 *"compressing the data using only symbols containing the CRS leads to significantly improved detection capacities"* 라고 적는다 — 대부분의 심볼이 비어 있어(부하가 낮아) 파일럿을 늘 담는 CRS 심볼만 쓰는 쪽이 오히려 나았다는 것이다.
> 
> 두 결과는 표준·표적·셀 부하가 서로 달라 어느 쪽도 다른 쪽을 반증하지 않는다. 이 리포트는 **상시 신호만 쓸 때의 기본선**을 재는 것으로 범위를 못박는다.

---
## §4. 우리가 쓴 방식 — Sionna PHY + 자작 격자, 그리고 검증

우리는 선행이 고른 그 상시 기준신호들을 **한 챔버에서 같은 조건으로** 나란히 세워 장단점을 가린다. 핵심은 **이미 있는 것은 다시 짜지 않는다**는 것이다:

- **5G NR 뉴머롤로지**(μ·SCS·슬롯·CP)는 🟢 **Sionna PHY `nr.CarrierConfig`** 에 물어 그대로 받는다(TS 38.211 구현) — 표를 손으로 짜지 않으니 규격과 어긋날 수 없고, 5G OFDM 파형 생성도 Sionna 가 한다. **LTE 뉴머롤로지**(15 kHz SCS·normal CP)는 🔴 TS 36.211 로 자작한다(nr.CarrierConfig 는 5G NR 전용).
- **WiFi(802.11) 파형과 상시 기준신호(CRS·SSB)의 자원격자 배치**만 🔴 3GPP TS 36.211/38.211·IEEE 802.11ac 표에서 자작한다(`src/waveforms.py`, 조사 근거 `docs/waveform_research.json`) — Sionna 가 이 부분을 기본 제공하지 않기 때문이다(§1).
- 그 격자에서 **점유대역 $B_{ref}$·반복주기(PRF)** 를 읽어 $\Delta R_b$=c/$B_{ref}$·$V_{b,max}$=PRF·λ/2 로 환산한다 — 거리·속도 두 축을 **같은 거리합 좌표**에 둔다(§2.2). 본문 숫자는 전부 이 측정에서 왔다.

**검증**은 이 파형이 정말 3GPP·IEEE 규격대로 만들어졌는지를 **→ report05** 에서 **Sionna PHY 로 교차대조**하는 것으로 한다 — 대역폭·뉴머롤로지·상시성을 규격 대비 확인한다. 즉 뉴머롤로지는 Sionna 가 준 값을 쓰고(§2), 자작 격자는 Sionna 로 되짚어 검증한다(report05).

> **정리** — 패시브가 빌릴 수 있는 조명은 '언제 봐도 똑같고(미리 앎), 늘 켜져 있는(상시)' 신호뿐이고, 그건 표준마다 하나뿐이다(§1). 그 하나의 넓이와 반복이 눈을 정하는데(§2), 넓고 자주 나오는 **LTE CRS 가 좁고 드문 5G SSB 보다 낫다**($\Delta R_b$ 16.7 vs 41.6 m, $V_{b,max}$ 81 vs 2.14 m/s). 선행도 같은 선택을 한다(§3). 뉴머롤로지는 Sionna PHY 에 물었고, 없는 WiFi·기준신호 격자만 규격표로 자작해 report05 에서 되짚는다(§4).

---
> **다음 리포트**: [report05](report05.ipynb) — 지금까지 '이 파형은 이런 대역·반복을 준다'고 했는데, 그 파형이 정말 3GPP·IEEE 규격대로 만들어졌는지 **Sionna PHY 로 교차대조**해 확인한다.